# IEEE-CIS Fraud Detection — Full Pipeline Runner

Runs EDA, all 7 model experiments, picks the winner, registers it in MLflow Model Registry, and generates submission.csv.

**Prerequisites:** Attach the `ieee-fraud-detection` competition dataset as input to this Kaggle notebook before running.

In [ ]:
import os, sys, subprocess, warnings, logging, gc
warnings.filterwarnings('ignore')
logging.getLogger('mlflow').setLevel(logging.ERROR)

subprocess.run(['pip', 'install', '-q', 'dagshub', 'mlflow', 'xgboost'], check=True)

REPO_DIR = '/kaggle/working/ML_Asgn2'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/Saba0033/ML_Asgn2.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from kaggle_secrets import UserSecretsClient
os.environ['DAGSHUB_USER_TOKEN'] = UserSecretsClient().get_secret('DAGSHUB_TOKEN')

import dagshub
dagshub.init(repo_owner='Saba0033', repo_name='ML_Asgn2', mlflow=True)
print('Bootstrap complete.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                               HistGradientBoostingClassifier)
from xgboost import XGBClassifier
from mlflow.models.signature import infer_signature

from src.data_utils import load_train, load_test, split_columns
from src.preprocessing import (build_linear_preprocessor, build_tree_preprocessor,
                                CorrelationPruner, engineer_features)
from src.mlflow_utils import (init_tracking, named_run, evaluate_classifier,
                               evaluate_train_val, cache_architecture_result)

RANDOM_STATE = 42
SAMPLE_FRAC = 0.3

X, y = load_train(sample_frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

engineer = FunctionTransformer(engineer_features, validate=False)
_eng_sample = engineer.transform(X_train.head(200))
num_cols, cat_cols = split_columns(_eng_sample)
del _eng_sample

print(f'Train: {len(X_train):,} rows, Val: {len(X_val):,} rows')
print(f'Features: {len(num_cols)} numeric, {len(cat_cols)} categorical')
print(f'Fraud rate: {y_train.mean():.4f}')

In [ ]:
sns.set_theme(style='whitegrid', context='talk')
PLOTS = 'plots'
os.makedirs(PLOTS, exist_ok=True)

fig, ax = plt.subplots(figsize=(6, 4))
vc = y.value_counts().sort_index()
ax.bar(['Legit (0)', 'Fraud (1)'], vc.values, color=['#4c72b0', '#c44e52'])
for i, v in enumerate(vc.values):
    ax.text(i, v, f'{v:,}\n({v/len(y):.2%})', ha='center', va='bottom')
ax.set_title('Class balance'); ax.set_ylabel('count')
fig.tight_layout(); fig.savefig(f'{PLOTS}/01_class_balance.png', dpi=120); plt.close()

missing = X.isna().mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(missing.values, bins=40, color='#dd8452', edgecolor='white')
ax.set_xlabel('fraction of NaN per column'); ax.set_ylabel('number of columns')
ax.set_title('Missing-value distribution across columns')
fig.tight_layout(); fig.savefig(f'{PLOTS}/02_missing_distribution.png', dpi=120); plt.close()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(X['TransactionAmt'].clip(upper=1000), bins=60, color='#4c72b0')
axes[0].set_title('TransactionAmt (clipped at 1000)'); axes[0].set_xlabel('amount')
axes[1].hist(np.log1p(X['TransactionAmt']), bins=60, color='#55a868')
axes[1].set_title('log1p(TransactionAmt)'); axes[1].set_xlabel('log amount')
fig.suptitle('Why we engineer TransactionAmt_log')
fig.tight_layout(); fig.savefig(f'{PLOTS}/03_transaction_amt.png', dpi=120); plt.close()

fig, ax = plt.subplots(figsize=(10, 5))
for cls, color, label in [(0, '#4c72b0', 'Legit'), (1, '#c44e52', 'Fraud')]:
    sns.kdeplot(np.log1p(X.loc[y == cls, 'TransactionAmt']), ax=ax,
                color=color, fill=True, alpha=0.4, label=label)
ax.set_xlabel('log1p(TransactionAmt)'); ax.set_title('log-amount distribution by class')
ax.legend(); fig.tight_layout(); fig.savefig(f'{PLOTS}/04_amt_by_class.png', dpi=120); plt.close()

for col in ['ProductCD', 'card4', 'card6', 'P_emaildomain']:
    if col not in X.columns: continue
    s = (pd.DataFrame({'cat': X[col].astype('object').fillna('missing'), 'y': y})
         .groupby('cat')['y'].agg(['mean', 'count'])
         .query('count >= 500').sort_values('mean', ascending=False).head(10))
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(s.index.astype(str), s['mean'], color='#c44e52')
    ax.axvline(y.mean(), color='black', ls='--', label=f'overall ({y.mean():.3f})')
    ax.set_xlabel('fraud rate'); ax.set_title(f'{col} by fraud rate (n>=500)'); ax.legend()
    fig.tight_layout(); fig.savefig(f'{PLOTS}/05_{col}_fraud_rate.png', dpi=120); plt.close()

hours = (X['TransactionDT'] / 3600) % 24
fig, ax = plt.subplots(figsize=(11, 4))
for cls, color, label in [(0, '#4c72b0', 'Legit'), (1, '#c44e52', 'Fraud')]:
    counts, _ = np.histogram(hours[y == cls], bins=24)
    ax.plot(range(24), counts / counts.sum(), color=color, label=label, lw=2)
ax.set_xlabel('hour-of-day'); ax.set_ylabel('share'); ax.set_title('Diurnal pattern')
ax.legend(); fig.tight_layout(); fig.savefig(f'{PLOTS}/06_hour_of_day.png', dpi=120); plt.close()

v_sample = [c for c in X.columns if c.startswith('V')][:30]
corr = X[v_sample].corr().fillna(0)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1, cbar_kws={'shrink': 0.6}, ax=ax)
ax.set_title('Correlation among first 30 V-features')
fig.tight_layout(); fig.savefig(f'{PLOTS}/07_v_correlation.png', dpi=120); plt.close()

print('EDA plots saved to plots/')

In [ ]:
def _selection_score(metrics):
    return metrics['val_roc_auc'] - 0.5 * max(0.0, metrics['overfit_gap'] - 0.02)

def _short_run_name(tag, params):
    abbrev = {
        'n_estimators': 'n', 'learning_rate': 'lr', 'max_depth': 'd',
        'min_samples_leaf': 'msl', 'subsample': 'ss', 'colsample_bytree': 'cs',
        'C': 'C', 'max_iter': 'iter', 'base_depth': 'bd', 'min_samples_split': 'mss',
        'max_leaf_nodes': 'mln', 'reg_alpha': 'ra', 'reg_lambda': 'rl',
    }
    parts = []
    for k, v in params.items():
        key = abbrev.get(k, k)
        val = str(v).replace('.', 'p').replace('None', 'NA')
        parts.append(f'{key}{val}')
    return f'{tag}_HP_' + '_'.join(parts)


def run_tree_experiment(model_tag, experiment_name, cache_name, hp_grid, make_estimator):
    print(f'\n{"="*60}\n  {model_tag}\n{"="*60}')
    init_tracking(experiment_name)

    cleaning_results = {}
    for fill in [-999.0, 0.0]:
        pre = build_tree_preprocessor(num_cols, cat_cols, numeric_fill=fill)
        probe = Pipeline([('eng', engineer), ('pre', pre),
                          ('clf', DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE))])
        probe.fit(X_train, y_train)
        m = evaluate_train_val(probe, X_train, y_train, X_val, y_val)
        cleaning_results[fill] = m
        with named_run(f'{model_tag}_Cleaning_fill{int(fill)}', tags={'stage':'cleaning'}):
            mlflow.log_param('numeric_fill', fill)
            mlflow.log_param('probe_model', 'DecisionTree(max_depth=6)')
            mlflow.log_metrics(m)
    best_fill = max(cleaning_results, key=lambda k: cleaning_results[k]['val_roc_auc'])
    preprocessor = build_tree_preprocessor(num_cols, cat_cols, numeric_fill=best_fill)
    print(f'  Cleaning: fill={best_fill}')

    probe = Pipeline([('eng', engineer), ('pre', preprocessor),
                      ('clf', DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE))])
    probe.fit(X_train, y_train)
    fe_m = evaluate_train_val(probe, X_train, y_train, X_val, y_val)
    with named_run(f'{model_tag}_FeatureEngineering', tags={'stage':'feature_engineering'}):
        mlflow.log_param('engineered', 'TransactionAmt_log + email TLDs')
        mlflow.log_metrics(fe_m)

    selectors = {
        'variance':    VarianceThreshold(threshold=0.0),
        'correlation': CorrelationPruner(threshold=0.95),
        'model_based': SelectFromModel(
            RandomForestClassifier(n_estimators=80, max_depth=8,
                                   n_jobs=-1, random_state=RANDOM_STATE),
            threshold='median'),
    }
    selection_results = {}
    for name, sel in selectors.items():
        pipe = Pipeline([('eng', engineer), ('pre', preprocessor), ('sel', sel),
                         ('clf', DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE))])
        pipe.fit(X_train, y_train)
        m = evaluate_train_val(pipe, X_train, y_train, X_val, y_val)
        m['selection_score'] = _selection_score(m)
        n_kept = int(np.asarray(sel.get_support()).sum())
        selection_results[name] = (sel, m, n_kept)
        with named_run(f'{model_tag}_FeatureSelection_{name}', tags={'stage':'feature_selection'}):
            mlflow.log_param('selector', name)
            mlflow.log_metric('n_features_kept', n_kept)
            loggable = {k: v for k, v in m.items() if isinstance(v, (int, float))}
            mlflow.log_metrics(loggable)
    best_sel_name = max(selection_results, key=lambda k: selection_results[k][1]['selection_score'])
    best_sel, _, best_sel_kept = selection_results[best_sel_name]
    print(f'  Selector: {best_sel_name} (kept={best_sel_kept})')

    results = []
    for params in hp_grid:
        estimator = make_estimator(params)
        pipe = Pipeline([('eng', engineer), ('pre', preprocessor),
                         ('sel', best_sel), ('clf', estimator)])
        pipe.fit(X_train, y_train)
        m = evaluate_train_val(pipe, X_train, y_train, X_val, y_val)
        m['selection_score'] = _selection_score(m)
        m['params'] = params
        results.append(m)
        with named_run(_short_run_name(model_tag, params), tags={'stage':'hp_tuning'}):
            mlflow.log_params(params)
            loggable = {k: v for k, v in m.items() if k != 'params' and isinstance(v, (int, float))}
            mlflow.log_metrics(loggable)
    best_idx = max(range(len(results)), key=lambda i: results[i]['selection_score'])
    best_params = results[best_idx]['params']
    best_tv = results[best_idx]
    print(f'  Best HP: {best_params}')
    print(f'    val_auc={best_tv["val_roc_auc"]:.4f}, gap={best_tv["overfit_gap"]:.4f}')

    best_pipe = Pipeline([('eng', engineer), ('pre', preprocessor),
                          ('sel', best_sel), ('clf', make_estimator(best_params))])
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    cv_s = cross_validate(best_pipe, X_train, y_train, cv=cv,
                           scoring=['roc_auc', 'average_precision'],
                           return_train_score=True, n_jobs=1)
    cv_summary = {
        'cv_train_roc_auc_mean': float(np.mean(cv_s['train_roc_auc'])),
        'cv_val_roc_auc_mean':   float(np.mean(cv_s['test_roc_auc'])),
        'cv_val_roc_auc_std':    float(np.std(cv_s['test_roc_auc'])),
        'cv_val_pr_auc_mean':    float(np.mean(cv_s['test_average_precision'])),
        'cv_overfit_gap':        float(np.mean(cv_s['train_roc_auc']) - np.mean(cv_s['test_roc_auc'])),
    }
    with named_run(f'{model_tag}_CrossValidation', tags={'stage':'cross_validation'}):
        mlflow.log_params(best_params)
        mlflow.log_metrics(cv_summary)
    print(f'  CV AUC: {cv_summary["cv_val_roc_auc_mean"]:.4f} +/- {cv_summary["cv_val_roc_auc_std"]:.4f}')

    X_full = pd.concat([X_train, X_val]); y_full = pd.concat([y_train, y_val])
    best_pipe.fit(X_full, y_full)
    fv = evaluate_classifier(best_pipe, X_val, y_val, prefix='final_val')
    sig = infer_signature(X_train.head(5), best_pipe.predict_proba(X_train.head(5)))
    with named_run(f'{model_tag}_FinalPipeline', tags={'stage':'final_pipeline'}) as fr:
        mlflow.log_params(best_params); mlflow.log_metrics(fv); mlflow.log_metrics(cv_summary)
        mlflow.sklearn.log_model(sk_model=best_pipe, name='pipeline',
                                  signature=sig, input_example=X_train.head(2))
        rid = fr.info.run_id

    cache_architecture_result(cache_name, {
        'best_params':         {k: (v if v is not None else 'None') for k, v in best_params.items()},
        'val_roc_auc':         float(best_tv['val_roc_auc']),
        'val_pr_auc':          float(best_tv['val_pr_auc']),
        'overfit_gap':         float(best_tv['overfit_gap']),
        'cv_val_roc_auc_mean': cv_summary['cv_val_roc_auc_mean'],
        'cv_val_roc_auc_std':  cv_summary['cv_val_roc_auc_std'],
        'cv_val_pr_auc_mean':  cv_summary['cv_val_pr_auc_mean'],
        'best_selector':       best_sel_name,
        'n_features_kept':     int(best_sel_kept),
        'final_run_id':        rid,
    })
    print(f'  DONE. run_id={rid}')
    gc.collect()
    return {'model_tag': model_tag, 'cache_name': cache_name,
            'cv_auc': cv_summary['cv_val_roc_auc_mean'], 'gap': best_tv['overfit_gap'],
            'run_id': rid, 'pipe': best_pipe}


def run_linear_experiment(model_tag, experiment_name, cache_name, penalty, hp_grid, make_estimator):
    print(f'\n{"="*60}\n  {model_tag}\n{"="*60}')
    init_tracking(experiment_name)

    probe_factory = lambda: LogisticRegression(penalty=penalty, solver='liblinear',
                                                C=1.0, max_iter=200, random_state=RANDOM_STATE)
    cleaning_results = {}
    for mc in [10, 20]:
        pre = build_linear_preprocessor(num_cols, cat_cols, max_categories=mc)
        probe = Pipeline([('eng', engineer), ('pre', pre), ('clf', probe_factory())])
        probe.fit(X_train, y_train)
        m = evaluate_train_val(probe, X_train, y_train, X_val, y_val)
        cleaning_results[mc] = m
        with named_run(f'{model_tag}_Cleaning_maxcats{mc}', tags={'stage':'cleaning'}):
            mlflow.log_param('max_categories', mc)
            mlflow.log_param('imputer', 'median + constant_missing')
            mlflow.log_param('scaler', 'StandardScaler')
            mlflow.log_metrics(m)
    best_mc = max(cleaning_results, key=lambda k: cleaning_results[k]['val_roc_auc'])
    preprocessor = build_linear_preprocessor(num_cols, cat_cols, max_categories=best_mc)
    print(f'  Cleaning: max_cats={best_mc}')

    probe = Pipeline([('eng', engineer), ('pre', preprocessor), ('clf', probe_factory())])
    probe.fit(X_train, y_train)
    fe_m = evaluate_train_val(probe, X_train, y_train, X_val, y_val)
    with named_run(f'{model_tag}_FeatureEngineering', tags={'stage':'feature_engineering'}):
        mlflow.log_param('engineered', 'TransactionAmt_log + email TLDs')
        mlflow.log_metrics(fe_m)

    selectors = {
        'variance': VarianceThreshold(threshold=0.001),
        'l1_logreg': SelectFromModel(
            LogisticRegression(penalty='l1', solver='liblinear', C=0.1,
                               max_iter=200, random_state=RANDOM_STATE),
            threshold='median'),
        'rf_importance': SelectFromModel(
            RandomForestClassifier(n_estimators=60, max_depth=8,
                                   n_jobs=-1, random_state=RANDOM_STATE),
            threshold='median'),
    }
    selection_results = {}
    for name, sel in selectors.items():
        pipe = Pipeline([('eng', engineer), ('pre', preprocessor),
                         ('sel', sel), ('clf', probe_factory())])
        pipe.fit(X_train, y_train)
        m = evaluate_train_val(pipe, X_train, y_train, X_val, y_val)
        m['selection_score'] = _selection_score(m)
        n_kept = int(np.asarray(sel.get_support()).sum())
        selection_results[name] = (sel, m, n_kept)
        with named_run(f'{model_tag}_FeatureSelection_{name}', tags={'stage':'feature_selection'}):
            mlflow.log_param('selector', name)
            mlflow.log_metric('n_features_kept', n_kept)
            loggable = {k: v for k, v in m.items() if isinstance(v, (int, float))}
            mlflow.log_metrics(loggable)
    best_sel_name = max(selection_results, key=lambda k: selection_results[k][1]['selection_score'])
    best_sel, _, best_sel_kept = selection_results[best_sel_name]
    print(f'  Selector: {best_sel_name} (kept={best_sel_kept})')

    results = []
    for params in hp_grid:
        estimator = make_estimator(params)
        pipe = Pipeline([('eng', engineer), ('pre', preprocessor),
                         ('sel', best_sel), ('clf', estimator)])
        pipe.fit(X_train, y_train)
        m = evaluate_train_val(pipe, X_train, y_train, X_val, y_val)
        m['selection_score'] = _selection_score(m)
        m['params'] = params
        results.append(m)
        with named_run(_short_run_name(model_tag, params), tags={'stage':'hp_tuning'}):
            mlflow.log_params(params)
            loggable = {k: v for k, v in m.items() if k != 'params' and isinstance(v, (int, float))}
            mlflow.log_metrics(loggable)
    best_idx = max(range(len(results)), key=lambda i: results[i]['selection_score'])
    best_params = results[best_idx]['params']
    best_tv = results[best_idx]
    print(f'  Best HP: {best_params}')
    print(f'    val_auc={best_tv["val_roc_auc"]:.4f}, gap={best_tv["overfit_gap"]:.4f}')

    best_pipe = Pipeline([('eng', engineer), ('pre', preprocessor),
                          ('sel', best_sel), ('clf', make_estimator(best_params))])
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    cv_s = cross_validate(best_pipe, X_train, y_train, cv=cv,
                           scoring=['roc_auc', 'average_precision'],
                           return_train_score=True, n_jobs=1)
    cv_summary = {
        'cv_train_roc_auc_mean': float(np.mean(cv_s['train_roc_auc'])),
        'cv_val_roc_auc_mean':   float(np.mean(cv_s['test_roc_auc'])),
        'cv_val_roc_auc_std':    float(np.std(cv_s['test_roc_auc'])),
        'cv_val_pr_auc_mean':    float(np.mean(cv_s['test_average_precision'])),
        'cv_overfit_gap':        float(np.mean(cv_s['train_roc_auc']) - np.mean(cv_s['test_roc_auc'])),
    }
    with named_run(f'{model_tag}_CrossValidation', tags={'stage':'cross_validation'}):
        mlflow.log_params(best_params)
        mlflow.log_metrics(cv_summary)
    print(f'  CV AUC: {cv_summary["cv_val_roc_auc_mean"]:.4f} +/- {cv_summary["cv_val_roc_auc_std"]:.4f}')

    X_full = pd.concat([X_train, X_val]); y_full = pd.concat([y_train, y_val])
    best_pipe.fit(X_full, y_full)
    fv = evaluate_classifier(best_pipe, X_val, y_val, prefix='final_val')
    sig = infer_signature(X_train.head(5), best_pipe.predict_proba(X_train.head(5)))
    with named_run(f'{model_tag}_FinalPipeline', tags={'stage':'final_pipeline'}) as fr:
        mlflow.log_params(best_params); mlflow.log_metrics(fv); mlflow.log_metrics(cv_summary)
        mlflow.sklearn.log_model(sk_model=best_pipe, name='pipeline',
                                  signature=sig, input_example=X_train.head(2))
        rid = fr.info.run_id

    cache_architecture_result(cache_name, {
        'best_params':         {k: (v if v is not None else 'None') for k, v in best_params.items()},
        'val_roc_auc':         float(best_tv['val_roc_auc']),
        'val_pr_auc':          float(best_tv['val_pr_auc']),
        'overfit_gap':         float(best_tv['overfit_gap']),
        'cv_val_roc_auc_mean': cv_summary['cv_val_roc_auc_mean'],
        'cv_val_roc_auc_std':  cv_summary['cv_val_roc_auc_std'],
        'cv_val_pr_auc_mean':  cv_summary['cv_val_pr_auc_mean'],
        'best_selector':       best_sel_name,
        'n_features_kept':     int(best_sel_kept),
        'final_run_id':        rid,
    })
    print(f'  DONE. run_id={rid}')
    gc.collect()
    return {'model_tag': model_tag, 'cache_name': cache_name,
            'cv_auc': cv_summary['cv_val_roc_auc_mean'], 'gap': best_tv['overfit_gap'],
            'run_id': rid, 'pipe': best_pipe}

print('Helper functions defined.')

In [ ]:
all_results = []

# 1. Logistic Regression L2
all_results.append(run_linear_experiment(
    'LogReg_L2', 'LogisticRegression_L2_Training', 'LogisticRegression_L2', 'l2',
    [{'C': 0.001}, {'C': 0.01}, {'C': 0.1}, {'C': 1.0}, {'C': 10.0}],
    lambda p: LogisticRegression(**p, penalty='l2', solver='liblinear',
                                  max_iter=300, random_state=RANDOM_STATE),
))

# 2. Logistic Regression L1
all_results.append(run_linear_experiment(
    'LogReg_L1', 'LogisticRegression_L1_Training', 'LogisticRegression_L1', 'l1',
    [{'C': 0.001}, {'C': 0.01}, {'C': 0.1}, {'C': 1.0}, {'C': 10.0}],
    lambda p: LogisticRegression(**p, penalty='l1', solver='liblinear',
                                  max_iter=300, random_state=RANDOM_STATE),
))

# 3. Decision Tree
all_results.append(run_tree_experiment(
    'DecisionTree', 'DecisionTree_Training', 'DecisionTree',
    [{'max_depth': 3}, {'max_depth': 6}, {'max_depth': 10}, {'max_depth': 16},
     {'max_depth': None, 'min_samples_leaf': 50}, {'max_depth': None, 'min_samples_leaf': 1}],
    lambda p: DecisionTreeClassifier(**p, random_state=RANDOM_STATE),
))

# 4. Random Forest
all_results.append(run_tree_experiment(
    'RandomForest', 'RandomForest_Training', 'RandomForest',
    [{'n_estimators': 50, 'max_depth': 4}, {'n_estimators': 100, 'max_depth': 8},
     {'n_estimators': 200, 'max_depth': 12}, {'n_estimators': 300, 'max_depth': 16},
     {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 20},
     {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 1}],
    lambda p: RandomForestClassifier(**p, n_jobs=-1, random_state=RANDOM_STATE),
))

# 5. AdaBoost
all_results.append(run_tree_experiment(
    'AdaBoost', 'AdaBoost_Training', 'AdaBoost',
    [{'n_estimators': 50, 'learning_rate': 1.0, 'base_depth': 1},
     {'n_estimators': 100, 'learning_rate': 1.0, 'base_depth': 1},
     {'n_estimators': 200, 'learning_rate': 0.5, 'base_depth': 3},
     {'n_estimators': 300, 'learning_rate': 0.1, 'base_depth': 5},
     {'n_estimators': 500, 'learning_rate': 0.5, 'base_depth': 8}],
    lambda p: AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=p.get('base_depth', 3)),
        n_estimators=p['n_estimators'], learning_rate=p['learning_rate'],
        random_state=RANDOM_STATE),
))

# 6. Gradient Boosting (HistGB)
all_results.append(run_tree_experiment(
    'GradientBoosting', 'GradientBoosting_Training', 'GradientBoosting',
    [{'learning_rate': 0.5, 'max_iter': 50, 'max_depth': 3},
     {'learning_rate': 0.1, 'max_iter': 100, 'max_depth': 4},
     {'learning_rate': 0.05, 'max_iter': 300, 'max_depth': 6},
     {'learning_rate': 0.05, 'max_iter': 500, 'max_depth': 8},
     {'learning_rate': 0.01, 'max_iter': 800, 'max_depth': 10},
     {'learning_rate': 0.5, 'max_iter': 800, 'max_depth': 12}],
    lambda p: HistGradientBoostingClassifier(**p, random_state=RANDOM_STATE),
))

# 7. XGBoost
all_results.append(run_tree_experiment(
    'XGBoost', 'XGBoost_Training', 'XGBoost',
    [{'n_estimators': 100, 'learning_rate': 0.3, 'max_depth': 3, 'subsample': 1.0, 'colsample_bytree': 1.0},
     {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 4, 'subsample': 0.9, 'colsample_bytree': 0.9},
     {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.8},
     {'n_estimators': 600, 'learning_rate': 0.05, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.7},
     {'n_estimators': 800, 'learning_rate': 0.03, 'max_depth': 10, 'subsample': 0.7, 'colsample_bytree': 0.6},
     {'n_estimators': 800, 'learning_rate': 0.3, 'max_depth': 12, 'subsample': 1.0, 'colsample_bytree': 1.0}],
    lambda p: XGBClassifier(**p, objective='binary:logistic', eval_metric='auc',
                              tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE),
))

print('\nAll 7 experiments complete.')

In [ ]:
import json

print('\n' + '='*70)
print('  RESULTS SUMMARY')
print('='*70)
for r in sorted(all_results, key=lambda x: x['cv_auc'], reverse=True):
    print(f"  {r['model_tag']:25s}  CV AUC={r['cv_auc']:.4f}  gap={r['gap']:.4f}")

winner = max(all_results, key=lambda r: r['cv_auc'])
print(f'\nWINNER: {winner["model_tag"]} (CV AUC={winner["cv_auc"]:.4f})')

init_tracking(f'{winner["cache_name"]}_Training')
mv = mlflow.register_model(
    model_uri=f'runs:/{winner["run_id"]}/pipeline',
    name='IEEEFraudBestModel',
)
print(f'Registered as IEEEFraudBestModel version {mv.version}')

X_test, transaction_ids = load_test()
print(f'\nTest: {len(X_test):,} rows')
fraud_proba = winner['pipe'].predict_proba(X_test)[:, 1]
os.makedirs('submissions', exist_ok=True)
submission = pd.DataFrame({'TransactionID': transaction_ids, 'isFraud': fraud_proba})
submission.to_csv('submissions/submission.csv', index=False)
print(f'Saved submissions/submission.csv ({len(submission):,} rows)')
print(submission.head())

if os.path.exists('results_cache.json'):
    print('\nresults_cache.json:')
    data = json.loads(open('results_cache.json').read())
    for arch in sorted(data, key=lambda a: data[a].get('cv_val_roc_auc_mean', 0), reverse=True):
        d = data[arch]
        print(f"  {arch:25s}  CV={d.get('cv_val_roc_auc_mean', 0):.4f}  gap={d.get('overfit_gap', 0):.4f}")

print('\nDone. Download submissions/submission.csv and submit to Kaggle:')
print('  kaggle competitions submit -c ieee-fraud-detection -f submissions/submission.csv -m "best model"')